# AEGES-Q: UNSW-NB15 QML Benchmark

## Objective

This notebook evaluates quantum machine-learning approaches for binary network intrusion detection using the UNSW-NB15 dataset.

The objective is to compare QML classification performance against the established classical Random Forest baseline under a consistent experimental setup.

### Evaluation Metrics

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- False Positive Rate (FPR)
- Training time
- Inference time

### Experimental Principle

The classical IDS pipeline has already been completed and is treated as the baseline.

The QML experiments use the same underlying UNSW-NB15 train/test split while constructing a quantum-compatible feature representation.

All experiments are performed locally using Qiskit Aer simulation.

## 1. Experimental Setup

In [18]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    mean_squared_error,
)

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

warnings.filterwarnings("ignore")

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Project root
PROJECT_ROOT = Path.cwd().parent.parent

print("Project root:", PROJECT_ROOT)
print("Python environment ready.")

Project root: c:\Projects\Aeges-Q
Python environment ready.


## 2. Load the UNSW-NB15 Dataset

The original author-provided training and testing split is retained:

- Training set: 82,332 samples
- Testing set: 175,341 samples

The existing classical preprocessing pipeline is not modified.

In [2]:
TRAIN_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "UNSW_NB15_training-set.csv"
)

TEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "UNSW_NB15_testing-set.csv"
)

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

print("\nTraining columns:")
print(train_df.columns.tolist())

print("\nTarget distribution:")
print(train_df["label"].value_counts().sort_index())

Training shape: (82332, 45)
Testing shape: (175341, 45)

Training columns:
['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']

Target distribution:
label
0    37000
1    45332
Name: count, dtype: int64


## 3. Load Existing Classical Preprocessing

In [4]:
import joblib

PREPROCESSOR_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "classical"
    / "variant_e_preprocessor.joblib"
)

preprocessor = joblib.load(PREPROCESSOR_PATH)

print("Loaded preprocessor:", type(preprocessor).__name__)
print("Transformed feature count:", len(preprocessor.get_feature_names_out()))

Loaded preprocessor: ColumnTransformer
Transformed feature count: 177


In [5]:
X_train_full = preprocessor.transform(
    train_df.drop(columns=["label"])
)

X_test_full = preprocessor.transform(
    test_df.drop(columns=["label"])
)

y_train_full = train_df["label"].to_numpy()
y_test_full = test_df["label"].to_numpy()

print("Processed training shape:", X_train_full.shape)
print("Processed testing shape:", X_test_full.shape)

Processed training shape: (82332, 177)
Processed testing shape: (175341, 177)


## 4. Quantum-Compatible Feature Reduction

The classical preprocessing pipeline produces 177 transformed features.

To make quantum simulation computationally practical, the transformed feature space is compressed into a small number of components using PCA.

Four principal components are used initially, corresponding to a four-qubit quantum feature map.

PCA is fitted only on the training data to prevent test-set information leakage.

In [6]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

# Convert sparse matrices if necessary
if hasattr(X_train_full, "toarray"):
    X_train_dense = X_train_full.toarray()
    X_test_dense = X_test_full.toarray()
else:
    X_train_dense = np.asarray(X_train_full)
    X_test_dense = np.asarray(X_test_full)

# PCA -> 4 quantum-compatible features
pca = PCA(
    n_components=4,
    random_state=RANDOM_STATE
)

X_train_pca = pca.fit_transform(X_train_dense)
X_test_pca = pca.transform(X_test_dense)

print("PCA training shape:", X_train_pca.shape)
print("PCA testing shape:", X_test_pca.shape)

print(
    "Explained variance ratio:",
    pca.explained_variance_ratio_
)

print(
    "Total explained variance:",
    pca.explained_variance_ratio_.sum()
)

PCA training shape: (82332, 4)
PCA testing shape: (175341, 4)
Explained variance ratio: [0.23131599 0.09122509 0.0777905  0.05604421]
Total explained variance: 0.45637578514427934


## 4.1 Encode PCA Features as Quantum Rotation Angles

In [7]:
# Scale PCA components using training data only
pca_scaler = MinMaxScaler(feature_range=(0, 1))

X_train_quantum = pca_scaler.fit_transform(X_train_pca)
X_test_quantum = pca_scaler.transform(X_test_pca)

# Map [0, 1] -> [0, pi]
X_train_angles = X_train_quantum * np.pi
X_test_angles = X_test_quantum * np.pi

print("Quantum training shape:", X_train_angles.shape)
print("Quantum testing shape:", X_test_angles.shape)

print("\nFirst encoded training sample:")
print(X_train_angles[0])

Quantum training shape: (82332, 4)
Quantum testing shape: (175341, 4)

First encoded training sample:
[0.61541876 1.33658505 0.33038435 0.6175476 ]


## 5. Quantum Circuit Candidates

Three variational quantum classifier architectures are evaluated:

- **Circuit A:** Feature encoding only
- **Circuit B:** Feature encoding + trainable rotation layer
- **Circuit C:** Feature encoding + trainable rotations + entanglement

All circuits use four qubits and the same four PCA-derived quantum features.

The objective is to determine whether additional trainable layers and qubit entanglement improve binary intrusion classification.

In [8]:
from qiskit.circuit import ParameterVector

N_QUBITS = 4

# Separate trainable parameters for each architecture
theta_b = ParameterVector("θB", length=4)
theta_c = ParameterVector("θC", length=8)


def build_circuit_a(features):
    """Circuit A: feature encoding only."""
    qc = QuantumCircuit(N_QUBITS, 1)

    for q in range(N_QUBITS):
        qc.ry(float(features[q]), q)

    qc.measure(0, 0)

    return qc


def build_circuit_b(features):
    """Circuit B: feature encoding + trainable rotations."""
    qc = QuantumCircuit(N_QUBITS, 1)

    # Feature encoding
    for q in range(N_QUBITS):
        qc.ry(float(features[q]), q)

    # Trainable layer
    for q in range(N_QUBITS):
        qc.ry(theta_b[q], q)

    qc.measure(0, 0)

    return qc


def build_circuit_c(features):
    """Circuit C: feature encoding + trainable rotations + entanglement."""
    qc = QuantumCircuit(N_QUBITS, 1)

    # Feature encoding
    for q in range(N_QUBITS):
        qc.ry(float(features[q]), q)

    # Trainable rotation layer
    for q in range(N_QUBITS):
        qc.ry(theta_c[q], q)

    # Entanglement
    for q in range(N_QUBITS - 1):
        qc.cx(q, q + 1)

    # Second trainable layer
    for q in range(N_QUBITS):
        qc.rz(theta_c[q + 4], q)

    qc.measure(0, 0)

    return qc

In [9]:
# Use one real dataset sample
example_features = X_train_angles[0]

circuit_a = build_circuit_a(example_features)
circuit_b = build_circuit_b(example_features)
circuit_c = build_circuit_c(example_features)

print("Circuit A:")
print(circuit_a)

print("\nCircuit B:")
print(circuit_b)

print("\nCircuit C:")
print(circuit_c)

Circuit A:
     ┌─────────────┐┌─┐
q_0: ┤ Ry(0.61542) ├┤M├
     └┬────────────┤└╥┘
q_1: ─┤ Ry(1.3366) ├─╫─
     ┌┴────────────┤ ║ 
q_2: ┤ Ry(0.33038) ├─╫─
     ├─────────────┤ ║ 
q_3: ┤ Ry(0.61755) ├─╫─
     └─────────────┘ ║ 
c: 1/════════════════╩═
                     0 

Circuit B:
     ┌─────────────┐┌───────────┐┌─┐
q_0: ┤ Ry(0.61542) ├┤ Ry(θB[0]) ├┤M├
     └┬────────────┤├───────────┤└╥┘
q_1: ─┤ Ry(1.3366) ├┤ Ry(θB[1]) ├─╫─
     ┌┴────────────┤├───────────┤ ║ 
q_2: ┤ Ry(0.33038) ├┤ Ry(θB[2]) ├─╫─
     ├─────────────┤├───────────┤ ║ 
q_3: ┤ Ry(0.61755) ├┤ Ry(θB[3]) ├─╫─
     └─────────────┘└───────────┘ ║ 
c: 1/═════════════════════════════╩═
                                  0 

Circuit C:
     ┌─────────────┐┌───────────┐     ┌───────────┐             ┌─┐»
q_0: ┤ Ry(0.61542) ├┤ Ry(θC[0]) ├──■──┤ Rz(θC[4]) ├─────────────┤M├»
     └┬────────────┤├───────────┤┌─┴─┐└───────────┘┌───────────┐└╥┘»
q_1: ─┤ Ry(1.3366) ├┤ Ry(θC[1]) ├┤ X ├──────■──────┤ Rz(θC[5]) ├─╫─»
     ┌┴───────────

## 6. Create the QML Benchmark Subset

To compare circuit architectures efficiently on the local simulator, a stratified subset is used for the initial benchmark.

The same subset is used for all candidate circuits to ensure a fair comparison.

The full UNSW-NB15 datasets remain unchanged and will be used for the final evaluation of the selected model.

In [13]:
from sklearn.model_selection import train_test_split

BENCHMARK_TRAIN_SIZE = 500
BENCHMARK_TEST_SIZE = 500

# Stratified benchmark training set
X_bench_train, _, y_bench_train, _ = train_test_split(
    X_train_angles,
    y_train_full,
    train_size=BENCHMARK_TRAIN_SIZE,
    stratify=y_train_full,
    random_state=RANDOM_STATE
)

# Stratified benchmark test set
X_bench_test, _, y_bench_test, _ = train_test_split(
    X_test_angles,
    y_test_full,
    train_size=BENCHMARK_TEST_SIZE,
    stratify=y_test_full,
    random_state=RANDOM_STATE
)

print("Benchmark training:", X_bench_train.shape)
print("Benchmark testing:", X_bench_test.shape)

Benchmark training: (500, 4)
Benchmark testing: (500, 4)


## 7. Quantum Circuit Evaluation Functions

In [14]:
from qiskit_aer import AerSimulator

simulator = AerSimulator()

print("Aer simulator initialized:", simulator)

Aer simulator initialized: AerSimulator('aer_simulator')


In [15]:
def run_quantum_probability(circuit, parameters=None, shots=256):
    """
    Execute a parameterized or fixed circuit and return
    the probability of measuring the output qubit as 1.
    """
    if parameters is not None:
        circuit = circuit.assign_parameters(parameters)

    result = simulator.run(
        circuit,
        shots=shots
    ).result()

    counts = result.get_counts()

    total = sum(counts.values())
    ones = counts.get("1", 0)

    return ones / total


def evaluate_circuit(
    circuit_builder,
    X_data,
    parameters=None,
    shots=256
):
    """
    Evaluate a circuit across a dataset.
    """
    probabilities = []

    for sample in X_data:
        circuit = circuit_builder(sample)

        probability = run_quantum_probability(
            circuit,
            parameters=parameters,
            shots=shots
        )

        probabilities.append(probability)

    return np.array(probabilities)

In [16]:
start_time = time.time()

circuit_a_probabilities = evaluate_circuit(
    build_circuit_a,
    X_bench_test,
    shots=256
)

circuit_a_time = time.time() - start_time

circuit_a_predictions = (
    circuit_a_probabilities >= 0.5
).astype(int)

circuit_a_accuracy = accuracy_score(
    y_bench_test,
    circuit_a_predictions
)

circuit_a_f1 = f1_score(
    y_bench_test,
    circuit_a_predictions,
    zero_division=0
)

circuit_a_auc = roc_auc_score(
    y_bench_test,
    circuit_a_probabilities
)

print(f"Circuit A evaluation time: {circuit_a_time:.2f}s")
print(f"Accuracy: {circuit_a_accuracy:.4f}")
print(f"F1-score: {circuit_a_f1:.4f}")
print(f"ROC-AUC: {circuit_a_auc:.4f}")

Circuit A evaluation time: 2.15s
Accuracy: 0.3200
F1-score: 0.0000
ROC-AUC: 0.2613


## 8. Benchmark Trainable Circuit B

Circuit B extends the basic feature-encoding circuit with a trainable single-qubit rotation layer.

Unlike Circuit A, Circuit B contains trainable parameters that can be optimized using the intrusion-detection training data.

The same benchmark training subset and evaluation procedure are used to maintain a fair comparison between circuit architectures.

In [19]:
# Initial parameters for Circuit B
theta_b_values = np.array([
    0.1, 0.2, 0.3, 0.4
], dtype=float)


def circuit_b_parameters(values):
    return {
        theta_b[i]: float(values[i])
        for i in range(4)
    }


def circuit_b_loss(values, X_data, y_data, shots=128):
    parameters = circuit_b_parameters(values)

    probabilities = evaluate_circuit(
        build_circuit_b,
        X_data,
        parameters=parameters,
        shots=shots
    )

    return mean_squared_error(y_data, probabilities)


print(
    "Initial Circuit B loss:",
    circuit_b_loss(
        theta_b_values,
        X_bench_train,
        y_bench_train
    )
)

Initial Circuit B loss: 0.4274459228515625


In [20]:
def train_circuit_b(
    initial_values,
    X_data,
    y_data,
    iterations=15,
    learning_rate=0.15,
    perturbation=0.10,
    shots=128
):
    values = initial_values.copy()
    history = []

    for iteration in range(iterations):

        direction = np.random.choice(
            [-1, 1],
            size=len(values)
        )

        plus_values = (
            values + perturbation * direction
        )

        minus_values = (
            values - perturbation * direction
        )

        plus_loss = circuit_b_loss(
            plus_values,
            X_data,
            y_data,
            shots
        )

        minus_loss = circuit_b_loss(
            minus_values,
            X_data,
            y_data,
            shots
        )

        gradient = (
            (plus_loss - minus_loss)
            / (2 * perturbation)
        ) * direction

        values -= learning_rate * gradient

        current_loss = circuit_b_loss(
            values,
            X_data,
            y_data,
            shots
        )

        history.append(current_loss)

        print(
            f"Iteration {iteration + 1:02d}/{iterations} "
            f"| Loss: {current_loss:.6f}"
        )

    return values, history

In [21]:
start_time = time.time()

trained_b_values, b_history = train_circuit_b(
    theta_b_values,
    X_bench_train,
    y_bench_train
)

circuit_b_training_time = time.time() - start_time

print(
    f"\nCircuit B training time: "
    f"{circuit_b_training_time:.2f}s"
)

print("Trained parameters:")
print(trained_b_values)

Iteration 01/15 | Loss: 0.417814
Iteration 02/15 | Loss: 0.413220
Iteration 03/15 | Loss: 0.407181
Iteration 04/15 | Loss: 0.398214
Iteration 05/15 | Loss: 0.389798
Iteration 06/15 | Loss: 0.388035
Iteration 07/15 | Loss: 0.382972
Iteration 08/15 | Loss: 0.374887
Iteration 09/15 | Loss: 0.373639
Iteration 10/15 | Loss: 0.369840
Iteration 11/15 | Loss: 0.369221
Iteration 12/15 | Loss: 0.360379
Iteration 13/15 | Loss: 0.355878
Iteration 14/15 | Loss: 0.353234
Iteration 15/15 | Loss: 0.350833

Circuit B training time: 91.55s
Trained parameters:
[0.50018149 0.23112619 0.41352859 0.41315311]


In [22]:
trained_b_parameters = circuit_b_parameters(
    trained_b_values
)

start_time = time.time()

circuit_b_probabilities = evaluate_circuit(
    build_circuit_b,
    X_bench_test,
    parameters=trained_b_parameters,
    shots=256
)

circuit_b_inference_time = time.time() - start_time

circuit_b_predictions = (
    circuit_b_probabilities >= 0.5
).astype(int)

print(f"Inference time: {circuit_b_inference_time:.2f}s")
print(
    f"Accuracy: "
    f"{accuracy_score(y_bench_test, circuit_b_predictions):.4f}"
)
print(
    f"Precision: "
    f"{precision_score(y_bench_test, circuit_b_predictions, zero_division=0):.4f}"
)
print(
    f"Recall: "
    f"{recall_score(y_bench_test, circuit_b_predictions, zero_division=0):.4f}"
)
print(
    f"F1: "
    f"{f1_score(y_bench_test, circuit_b_predictions, zero_division=0):.4f}"
)
print(
    f"ROC-AUC: "
    f"{roc_auc_score(y_bench_test, circuit_b_probabilities):.4f}"
)

Inference time: 2.37s
Accuracy: 0.3640
Precision: 0.6310
Recall: 0.1559
F1: 0.2500
ROC-AUC: 0.2660


## 8.1 Circuit B Results

Circuit B reduced the training loss from approximately 0.4274 to 0.3508 over 15 SPSA iterations.

However, its test performance remained poor, with an accuracy of 36.4%, F1-score of 0.25, and ROC-AUC of 0.2660.

The low ROC-AUC indicates that the learned output probability does not provide useful separation between normal and attack traffic.

Therefore, Circuit B is not selected as the final QML classifier. The result motivates testing an entangling architecture in Circuit C.